# Simple AAAI Nesy-Gen Experiment Notebook

This notebook runs the main paper pipeline without exposing every ablation switch.

Pipeline: dataset manifest -> radiology PrimeKG cache -> entity-linking validation -> Vision-T5 training -> RAG + PrimeKG/LTN/Consistency Gate generation -> evaluation -> official metric input export.

Only change Cell 2 for normal use.

In [ ]:
# Cell 1: Setup repository and dependencies
from pathlib import Path

REPO_DIR = Path('/content/Nesy-GenRevision')
if REPO_DIR.exists():
    !git -C /content/Nesy-GenRevision pull --ff-only
else:
    !git clone https://github.com/FaezehMillerAI/Nesy-GenRevision.git /content/Nesy-GenRevision

%cd /content/Nesy-GenRevision
!pip -q install kagglehub transformers accelerate sentencepiece timm scikit-learn rouge-score nltk tqdm pandas pillow

In [ ]:
# Cell 2: Main configuration
from pathlib import Path

# Choose one dataset.
RUN_DATASET = 'iuxray_official'  # options: 'iuxray_official', 'mimic_aug'

# Start with 'smoke'. When it works, change to 'full'.
RUN_SIZE = 'smoke'  # options: 'smoke', 'full'

# Main method switch.
USE_GRAPH_CONSTRAINTS = True
# WOW setup: graph-aware token training plus graph-constrained RAG selection.
# Leakage is guarded in retrieval by filtering same-study and exact duplicate-reference candidates.
USE_GRAPH_AWARE_TRAINING = True

# Drive paths.
IUXRAY_ROOT = Path('/content/drive/MyDrive/iuxray')
PRIMEKG_SOURCE_DIR = Path('/content/drive/MyDrive/dataverse_files')
EXPERIMENT_ROOT = Path('/content/drive/MyDrive/aaai_2026_experiments_simple')

# Recommended stable model for A100. The visual backbone remains frozen by default.
TEXT_MODEL_NAME = 'google/flan-t5-base'
TOKENIZER_NAME = TEXT_MODEL_NAME
VISION_BACKBONE = 'convnext_base'
FREEZE_VISUAL_ENCODER = True

SEED = 13
RUN_NAME = 'nesygen_simple_graph' if USE_GRAPH_CONSTRAINTS else 'nesygen_simple_standard'
OUTPUT_DIR = EXPERIMENT_ROOT / RUN_DATASET / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if RUN_SIZE == 'smoke':
    EPOCHS = 1
    MAX_TRAIN_EXAMPLES = 256
    MAX_VAL_EXAMPLES = 64
    MAX_TEST_EXAMPLES = 100
    TRAIN_BATCH_SIZE = 4
else:
    EPOCHS = 6
    MAX_TRAIN_EXAMPLES = None
    MAX_VAL_EXAMPLES = None
    MAX_TEST_EXAMPLES = None
    TRAIN_BATCH_SIZE = 16

LEARNING_RATE = 5e-5
MAX_TARGET_LENGTH = 192
IMAGE_SIZE = 224
NUM_WORKERS = 4
USE_BF16 = True
COMPILE_MODEL = True

# Generation settings. These reduce repeated templates while keeping evidence and graph checks.
# The default profile is BLEU-guarded graph selection: evidence-heavy but still PrimeKG/LTN verified.
RETRIEVAL_TOP_K = 20
GENERATOR_NUM_CANDIDATES = 8
GENERATOR_NUM_BEAMS = 8
GENERATOR_NUM_BEAM_GROUPS = 4
GENERATOR_DIVERSITY_PENALTY = 0.35
GENERATOR_REPETITION_PENALTY = 1.12
GENERATOR_NO_REPEAT_NGRAM_SIZE = 3
MAX_NEW_TOKENS = 140

print('Dataset:', RUN_DATASET)
print('Run size:', RUN_SIZE)
print('Graph constraints:', USE_GRAPH_CONSTRAINTS)
print('Graph-aware training:', USE_GRAPH_AWARE_TRAINING)
print('Output:', OUTPUT_DIR)

In [ ]:
# Cell 3: Resolve dataset root
import kagglehub
from pathlib import Path

if RUN_DATASET == 'iuxray_official':
    DATA_ROOT = IUXRAY_ROOT
    if not (DATA_ROOT / 'annotation.json').exists():
        raise FileNotFoundError('Expected /content/drive/MyDrive/iuxray/annotation.json')
elif RUN_DATASET == 'mimic_aug':
    DATA_ROOT = Path(kagglehub.dataset_download('simhadrisadaram/mimic-cxr-dataset'))
    if not (DATA_ROOT / 'mimic_cxr_aug_train.csv').exists():
        raise FileNotFoundError(f'MIMIC augmented CSVs not found under {DATA_ROOT}')
else:
    raise ValueError(RUN_DATASET)

print('Data root:', DATA_ROOT)
print('Exists:', DATA_ROOT.exists())

In [ ]:
# Cell 4: Small command runner
import os
import subprocess
import sys

def run_cmd(cmd, cwd=REPO_DIR):
    cmd = [str(x) for x in cmd]
    print('\n$ ' + ' '.join(cmd))
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'Command failed with exit code {code}')
    return code

In [ ]:
# Cell 5: Build the dataset manifest
MANIFEST = OUTPUT_DIR / f'{RUN_DATASET}_manifest.jsonl'

run_cmd([
    sys.executable, 'scripts/build_manifest.py',
    '--dataset', RUN_DATASET,
    '--data-root', DATA_ROOT,
    '--output', MANIFEST,
    '--seed', SEED,
])

print('Manifest:', MANIFEST)
print('Exists:', MANIFEST.exists())

In [ ]:
# Cell 6: Build or reuse the radiology PrimeKG cache
RAD_PRIMEKG_DIR = Path(f'/content/drive/MyDrive/primekg_radiology_cache_{RUN_DATASET}')
RAD_PRIMEKG_DIR.mkdir(parents=True, exist_ok=True)

kg_cached = (RAD_PRIMEKG_DIR / 'kg.csv').exists() and (RAD_PRIMEKG_DIR / 'nodes.csv').exists()
if kg_cached:
    print('Using cached radiology PrimeKG:', RAD_PRIMEKG_DIR)
else:
    run_cmd([
        sys.executable, 'scripts/build_radiology_primekg.py',
        '--primekg-dir', PRIMEKG_SOURCE_DIR,
        '--manifest', MANIFEST,
        '--output-dir', RAD_PRIMEKG_DIR,
        '--hops', 1,
    ])

print('PrimeKG kg:', RAD_PRIMEKG_DIR / 'kg.csv')
print('PrimeKG nodes:', RAD_PRIMEKG_DIR / 'nodes.csv')

In [ ]:
# Cell 7: Reviewer-facing entity extraction/linking validation bundle
ENTITY_VALIDATION_DIR = OUTPUT_DIR / 'entity_linking_validation'
entity_limit = 300 if RUN_SIZE == 'smoke' else None

cmd = [
    sys.executable, 'scripts/validate_entity_linking.py',
    '--manifest', MANIFEST,
    '--nodes-csv', RAD_PRIMEKG_DIR / 'nodes.csv',
    '--output-dir', ENTITY_VALIDATION_DIR,
    '--prefix', RUN_DATASET,
    '--split', 'test',
    '--audit-sample-size', 100,
    '--seed', SEED,
]
if entity_limit:
    cmd += ['--limit', entity_limit]

run_cmd(cmd)
print('Validation bundle:', ENTITY_VALIDATION_DIR)

In [ ]:
# Cell 8: Train the frozen-vision Vision-T5 generator
CKPT_DIR = OUTPUT_DIR / 'checkpoints' / f'vision_t5_{VISION_BACKBONE}_{TEXT_MODEL_NAME.split("/")[-1]}'

cmd = [
    sys.executable, '-u', 'scripts/train_vision_t5_generator.py',
    '--manifest', MANIFEST,
    '--output-dir', CKPT_DIR,
    '--text-model-name', TEXT_MODEL_NAME,
    '--tokenizer-name', TOKENIZER_NAME,
    '--visual-backbone', VISION_BACKBONE,
    '--image-size', IMAGE_SIZE,
    '--epochs', EPOCHS,
    '--batch-size', TRAIN_BATCH_SIZE,
    '--gradient-accumulation-steps', 1,
    '--learning-rate', LEARNING_RATE,
    '--max-target-length', MAX_TARGET_LENGTH,
    '--num-workers', NUM_WORKERS,
    '--progress-style', 'tqdm',
    '--seed', SEED,
]

if FREEZE_VISUAL_ENCODER:
    cmd += ['--freeze-visual-encoder']
if USE_BF16:
    cmd += ['--bf16']
else:
    cmd += ['--fp16']
if COMPILE_MODEL:
    cmd += ['--compile-model']
if MAX_TRAIN_EXAMPLES:
    cmd += ['--max-train-examples', MAX_TRAIN_EXAMPLES]
if MAX_VAL_EXAMPLES:
    cmd += ['--max-val-examples', MAX_VAL_EXAMPLES]
if USE_GRAPH_AWARE_TRAINING:
    cmd += [
        '--graph-training-mode', 'primekg_token',
        '--graph-loss-nodes-csv', RAD_PRIMEKG_DIR / 'nodes.csv',
        '--graph-token-loss-weight', 0.05,
        '--unsupported-token-loss-weight', 0.0,
        '--graph-loss-source', 'indication_reference',
    ]

run_cmd(cmd)
print('Checkpoint:', CKPT_DIR)

In [ ]:
# Cell 9: Generate reports with RAG + PrimeKG LTN + Consistency Gate
GENERATOR_NAME = 'vision_t5_rag_primekg_graph' if USE_GRAPH_CONSTRAINTS else 'vision_t5_rag_primekg_standard'
PREDICTIONS_CSV = OUTPUT_DIR / f'{RUN_DATASET}_{GENERATOR_NAME}_predictions.csv'
CANDIDATES_CSV = OUTPUT_DIR / f'{RUN_DATASET}_{GENERATOR_NAME}_candidate_audit.csv'
DECODING_MODE = 'graph_constrained' if USE_GRAPH_CONSTRAINTS else 'standard'

cmd = [
    sys.executable, 'scripts/generate_rag_primekg_reports.py',
    '--manifest', MANIFEST,
    '--primekg-dir', RAD_PRIMEKG_DIR,
    '--output-csv', PREDICTIONS_CSV,
    '--candidates-csv', CANDIDATES_CSV,
    '--split', 'test',
    '--retrieval-top-k', RETRIEVAL_TOP_K,
    '--generator-checkpoint-dir', CKPT_DIR,
    '--generator-num-candidates', GENERATOR_NUM_CANDIDATES,
    '--generator-num-beams', GENERATOR_NUM_BEAMS,
    '--generator-batch-size', 2,
    '--generator-repetition-penalty', GENERATOR_REPETITION_PENALTY,
    '--generator-no-repeat-ngram-size', GENERATOR_NO_REPEAT_NGRAM_SIZE,
    '--generator-num-beam-groups', GENERATOR_NUM_BEAM_GROUPS,
    '--generator-diversity-penalty', GENERATOR_DIVERSITY_PENALTY,
    '--max-new-tokens', MAX_NEW_TOKENS,
    '--decoding-mode', DECODING_MODE,
    '--selection-objective', 'hybrid',
    '--graph-score-weight', 0.30,
    '--evidence-weight', 0.60,
    '--gate-weight', 0.10,
    '--generated-evidence-score', 0.40,
]
if MAX_TEST_EXAMPLES:
    cmd += ['--limit', MAX_TEST_EXAMPLES]

run_cmd(cmd)
print('Predictions:', PREDICTIONS_CSV)
print('Candidate audit:', CANDIDATES_CSV)

In [ ]:
# Cell 10: Evaluate lexical quality, graph/entity factuality proxy, and quick clinical metrics
EVAL_JSON = OUTPUT_DIR / f'{RUN_DATASET}_{GENERATOR_NAME}_quick_eval.json'

run_cmd([
    sys.executable, 'scripts/evaluate_generation.py',
    '--manifest', MANIFEST,
    '--predictions-csv', PREDICTIONS_CSV,
    '--output-json', EVAL_JSON,
    '--nodes-csv', RAD_PRIMEKG_DIR / 'nodes.csv',
    '--output-factuality-csv', OUTPUT_DIR / f'{RUN_DATASET}_{GENERATOR_NAME}_entity_factuality.csv',
    '--output-chexpert-csv', OUTPUT_DIR / f'{RUN_DATASET}_{GENERATOR_NAME}_chexpert_quick.csv',
    '--output-radgraph-csv', OUTPUT_DIR / f'{RUN_DATASET}_{GENERATOR_NAME}_radgraph_quick.csv',
])

print('Evaluation JSON:', EVAL_JSON)

In [ ]:
# Cell 11: Leakage audit and official metric input export
LEAKAGE_JSON = OUTPUT_DIR / f'{RUN_DATASET}_{GENERATOR_NAME}_leakage_audit.json'
OFFICIAL_INPUT_DIR = OUTPUT_DIR / 'official_metric_inputs'

run_cmd([
    sys.executable, 'scripts/audit_prediction_leakage.py',
    '--manifest', MANIFEST,
    '--predictions-csv', PREDICTIONS_CSV,
    '--output-json', LEAKAGE_JSON,
])

run_cmd([
    sys.executable, 'scripts/evaluate_generation.py',
    '--manifest', MANIFEST,
    '--predictions-csv', PREDICTIONS_CSV,
    '--output-json', OUTPUT_DIR / f'{RUN_DATASET}_{GENERATOR_NAME}_official_input_prep.json',
    '--prepare-official-inputs-dir', OFFICIAL_INPUT_DIR,
])

print('Leakage audit:', LEAKAGE_JSON)
print('Official metric inputs:', OFFICIAL_INPUT_DIR)
print('Next: run official CheXbert/CheXpert and RadGraph tools on these exported files, then feed their outputs back to scripts/evaluate_generation.py with --official-only.')

In [ ]:
# Cell 12: Inspect qualitative examples
import pandas as pd
from IPython.display import display

preds = pd.read_csv(PREDICTIONS_CSV)
cols = [c for c in ['study_id', 'source', 'graph_score', 'evidence_score', 'gate_decision', 'prediction', 'reference'] if c in preds.columns]
display(preds[cols].head(10))

print('Prediction diversity:')
print('unique predictions:', preds['prediction'].nunique(), '/', len(preds))
print('\nTop repeated predictions:')
display(preds['prediction'].value_counts().head(5).to_frame('count'))